In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from ipywidgets import interact, Dropdown, IntSlider, SelectionRangeSlider


In [26]:
for csv_name, key in [('VN30_INDEX.csv', 'VN30 Index'), ('VN_INDEX.csv', 'VN Index')]:
    df_temp = pd.read_csv(f'../dataset/{csv_name}')
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=True, errors='coerce')
    df_temp = df_temp.sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{key}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[key] = df_temp_filtered.set_index('time')[ret_col] * 100

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f'../dataset/{file}')
    if 'Date' in df_temp.columns:
        df_temp.rename(columns={'Date': 'time'}, inplace=True)
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=False, errors='coerce')
    df_temp = df_temp.dropna(subset=['time']).sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{file}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[file.replace('.csv', '')] = df_temp_filtered.set_index('time')[ret_col] * 100


In [24]:
summary_split = pd.DataFrame({
    "Dataset": [
        "VN30 Index", "VN Index", "DAX_40", "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI", "snp500", "Nikkei_225"
    ],
    "train_size": [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "val_size":   [1096, 1103, 1104, 1115, 1362, 1109, 1135, 1109, 1174],
    "test_size":  [925, 925, 972, 979, 923, 881, 901, 927, 1062],
    "split_i":    [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "split_j":    [3067, 3067, 3087, 3120, 3177, 3053, 3122, 3097, 2851]
})
split_df = summary_split.set_index("Dataset")[["train_size", "val_size", "test_size"]]
split_df.columns = ["Train", "Val", "Test"]
split_df

,Train,Val,Test
Dataset,,,
VN30 Index,1971,1096,925
VN Index,1964,1103,925
DAX_40,1983,1104,972
EuroNext_100,2005,1115,979
IBEX_35,1815,1362,923
KOSPI_index,1944,1109,881
SMI,1987,1135,901
snp500,1988,1109,927
Nikkei_225,1677,1174,1062


In [30]:
def calc_vol(x, typ, window):
    if typ == "abs(return)":
        return x.abs()
    if typ == "vol(rolling)":
        return x.rolling(window).std()
    return x.rolling(window).apply(lambda y: (y**2).mean()**0.5, raw=True)

def plot_chart(ds, chart_type, typ, period, window):
    try:
        series = datasets[ds]
        tr, va, te = map(int, split_df.loc[ds, ['Train', 'Val', 'Test']])
        fig = go.Figure()
        if chart_type == "Volatility":
            full_v = calc_vol(series, typ, window)
            if period == 'all':
                for s, e, c, n in [(0, tr, '#1f77b4', 'train'), (tr, tr+va, '#ff7f0e', 'val'), (tr+va, tr+va+te, '#2ca02c', 'test')]:
                    fig.add_trace(go.Scatter(x=series.index[s:e], y=full_v.iloc[s:e], mode="lines", name=n, line=dict(color=c), connectgaps=False))
            else:
                s, e = (0, tr) if period == 'train' else (tr, tr+va) if period == 'val' else (tr+va, tr+va+te)
                fig.add_trace(go.Scatter(x=series.index[s:e], y=full_v.iloc[s:e], mode="lines", name=period, line=dict(color="#1f77b4"), connectgaps=False))
            fig.update_layout(title=f"{ds} - {typ} ({period}, window={window})", xaxis_title="Time", yaxis_title="Volatility")
        else:
            if period == 'all':
                for s, e, n in [(0, tr, 'train'), (tr, tr+va, 'val'), (tr+va, tr+va+te, 'test')]:
                    fig.add_trace(go.Histogram(x=series.iloc[s:e], nbinsx=50, name=n))
            else:
                s, e = (0, tr) if period == 'train' else (tr, tr+va) if period == 'val' else (tr+va, tr+va+te)
                fig.add_trace(go.Histogram(x=series.iloc[s:e], nbinsx=50, name=ds))
            fig.update_layout(title=f"{ds} - Return Distribution ({period})", xaxis_title="Return (%)", yaxis_title="Frequency", xaxis=dict(range=[-16, 16]))
        fig.update_layout(height=500)
        fig.show()
    except Exception as e:
        print(f"Error: {e}")

ds_opts = [k for k in split_df.index if k in datasets]
interact(
    plot_chart,
    ds=Dropdown(options=ds_opts),
    chart_type=Dropdown(options=["Volatility", "Histogram"], value="Volatility"),
    typ=Dropdown(options=["abs(return)", "vol(rolling)", "realized_vol"], value="vol(rolling)"),
    period=Dropdown(options=["train", "val", "test", "all"], value="train"),
    window=Dropdown(options=[10, 20, 30, 50, 60, 120, 250], value=60)
)

interactive(children=(Dropdown(description='ds', options=('VN30 Index', 'VN Index', 'DAX_40', 'EuroNext_100', …

<function __main__.plot_chart(ds, chart_type, typ, period, window)>